In [ ]:
# ============================================================
# 셀1 - 환경 확인 및 캐시 삭제
# ============================================================
import os
import sys
import torch
import ultralytics
from pathlib import Path

print(f"현재 환경 경로  : {sys.executable}")
print(f"Python 버전    : {sys.version}")
print(f"PyTorch 버전   : {torch.__version__}")
print(f"Ultralytics 버전: {ultralytics.__version__}")
print(f"CUDA 사용 가능  : {torch.cuda.is_available()}")
print(f"가용 GPU 개수   : {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU 장치명      : {torch.cuda.get_device_name(0)}")
    print(f"CUDA 버전       : {torch.version.cuda}")
else:
    print("[경고] GPU를 인식할 수 없습니다. 학습 전 환경을 확인하십시오.")

# 캐시 삭제
dataset_root = r"C:\Users\SSAFY\Desktop\03\datasets\person.v2i.yolov11"
for split in ["train", "valid", "test"]:
    cache = Path(dataset_root) / split / "labels.cache"
    if cache.exists():
        cache.unlink()
        print(f"[캐시 삭제] {cache}")
    else:
        print(f"[캐시 없음] {cache}")

In [1]:
# ============================================================
# 셀2 - 경로 설정 및 data.yaml 업데이트
# ============================================================
import os
import yaml
from pathlib import Path

dataset_root = r"C:\Users\SSAFY\Desktop\03\datasets\person.v2i.yolov11"
yaml_path    = os.path.join(dataset_root, "data.yaml")
project_dir  = r"C:\Users\SSAFY\Desktop\03\models"

# 매번 직접 수정 - 규칙: yolo(모델버전)_(이미지크기)_(학습버전)
run_name = "yolo11n_480_v2"
print(f"이번 학습 버전: {run_name}")

# data.yaml 업데이트
with open(yaml_path, 'r', encoding='utf-8') as f:
    data = yaml.safe_load(f)

data['path']  = dataset_root
data['train'] = "train/images"
data['val']   = "valid/images"
data['test']  = "test/images"
data['nc']    = 1
data['names'] = ['person']

with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data, f, allow_unicode=True)

print(f"--- data.yaml 업데이트 완료 ---")
print(yaml.dump(data, allow_unicode=True))

# 데이터셋 통계 출력
for split in ['train', 'valid', 'test']:
    img_dir = Path(dataset_root) / split / 'images'
    if img_dir.exists():
        count = len(list(img_dir.glob('*.*')))
        print(f"[{split:5s}] 이미지 수: {count}장")
    else:
        print(f"[{split:5s}] 경로 없음: {img_dir}")

이번 학습 버전: yolo11n_480_v2
--- data.yaml 업데이트 완료 ---
names:
- person
nc: 1
path: C:\Users\SSAFY\Desktop\03\datasets\person.v2i.yolov11
roboflow:
  license: CC BY 4.0
  project: person-t8ttp
  url: https://universe.roboflow.com/swanut97/person-t8ttp/dataset/2
  version: 2
  workspace: swanut97
test: test/images
train: train/images
val: valid/images

[train] 이미지 수: 10392장
[valid] 이미지 수: 434장
[test ] 이미지 수: 433장


In [ ]:
# ============================================================
# 셀3 - 학습
# ============================================================
from ultralytics import YOLO

# 이어서 학습: RESUME = True
# 새로 학습  : RESUME = False
RESUME = True

if RESUME:
    model = YOLO(os.path.join(project_dir, run_name, "weights", "last.pt"))
else:
    model = YOLO("yolo11n.pt")

model.train(
    data=yaml_path,
    epochs=100,
    imgsz=480,
    # RTX 4050 Laptop (6GB) 기준 - OOM 발생 시 8로 낮출 것
    batch=16,
    patience=30,
    device=0,
    workers=4,
    project=project_dir,
    name=run_name,
    exist_ok=True,
    optimizer='AdamW',
    amp=True,
    # Roboflow에서 증강 적용 완료 - 중복 증강 최소화
    mosaic=1.0,       # mosaic은 Roboflow 증강과 성격이 달라 유지
    mixup=0.0,
    copy_paste=0.0,
    degrees=15.0,
    fliplr=0.5,
    hsv_v=0.4,
    hsv_h=0.015,
    hsv_s=0.7,
    translate=0.1,
    scale=0.3,
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    save=True,
    save_period=10,
    resume=RESUME,
    verbose=True
)

print(f"\n{'='*50}")
print(f"학습 완료: {run_name}")
print(f"저장 경로: {project_dir}\\{run_name}\\")
print(f"{'='*50}")

In [ ]:
# ============================================================
# 셀4 - 학습 결과 확인 (loss 그래프 + confusion matrix)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

result_dir = Path(project_dir) / run_name

# loss / mAP 그래프
results_png = result_dir / "results.png"
if results_png.exists():
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.imshow(mpimg.imread(results_png))
    ax.axis('off')
    ax.set_title(f"{run_name} - Training Results", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print(f"[경고] results.png 없음: {results_png}")

# confusion matrix
conf_matrix = result_dir / "confusion_matrix_normalized.png"
if conf_matrix.exists():
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(mpimg.imread(conf_matrix))
    ax.axis('off')
    ax.set_title(f"{run_name} - Confusion Matrix (Normalized)", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print(f"[경고] confusion_matrix_normalized.png 없음: {conf_matrix}")

In [ ]:
# ============================================================
# 셀5 - val set 추론 샘플 시각화
# ============================================================
import random
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from ultralytics import YOLO

best_model_path = Path(project_dir) / run_name / "weights" / "best.pt"
val_img_dir     = Path(dataset_root) / "valid" / "images"

if not best_model_path.exists():
    print(f"[ERROR] 모델 없음: {best_model_path}")
else:
    model = YOLO(str(best_model_path))
    img_paths = list(val_img_dir.glob("*.jpg")) + list(val_img_dir.glob("*.png"))

    # 랜덤 샘플 9장
    samples = random.sample(img_paths, min(9, len(img_paths)))

    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    fig.suptitle(f"{run_name} - Val Set Sample Predictions", fontsize=16)

    for ax, img_path in zip(axes.flatten(), samples):
        results = model.predict(str(img_path), imgsz=480, conf=0.25, verbose=False)
        result_img = results[0].plot()
        result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
        ax.imshow(result_img)
        ax.axis('off')
        ax.set_title(f"{img_path.name[:20]}...", fontsize=8)

    plt.tight_layout()
    plt.show()
    print(f"샘플 추론 완료 ({len(samples)}장)")

In [ ]:
# ============================================================
# 셀6 - ONNX 변환
# ============================================================
import os
import shutil
from ultralytics import YOLO
from pathlib import Path

best_model_path = Path(project_dir) / run_name / "weights" / "best.pt"
dst_dir         = Path(project_dir) / run_name / "weights" / "onnx"

if not best_model_path.exists():
    print(f"[ERROR] 모델 없음: {best_model_path}")
else:
    os.makedirs(dst_dir, exist_ok=True)

    # 작업 공간 분리 (원본 보존)
    isolated_pt = dst_dir / "best.pt"
    shutil.copy2(best_model_path, isolated_pt)

    model = YOLO(str(isolated_pt))

    print(f"\n{'='*50}")
    print(f"ONNX 변환 시작... ({run_name})")
    print(f"{'='*50}\n")

    model.export(
        format="onnx",
        imgsz=480,
        opset=17,          # TensorRT 변환 호환성
        simplify=True,     # 모델 단순화
        dynamic=False,     # Orin Nano 고정 배치 추론
        half=False,
    )

    # 결과 파일 정리
    # Ultralytics는 .pt 파일 위치에 .onnx를 저장함
    onnx_src = isolated_pt.with_suffix('.onnx')
    if onnx_src.exists():
        print(f"\n{'='*50}")
        print(f"변환 완료: {run_name}")
        print(f"저장 경로: {dst_dir}")
        print(f"파일 크기: {onnx_src.stat().st_size / (1024*1024):.2f} MB")
        print(f"{'='*50}")
        print(f"\n[다음 단계] Orin Nano에서 TensorRT engine 변환:")
        print(f"  trtexec --onnx=best.onnx --saveEngine=best.engine --int8 --workspace=512")
    else:
        print(f"[ERROR] ONNX 파일 생성 실패")

[다음 단계] Orin Nano에서 TensorRT engine 변환:

trtexec --onnx=best.onnx --saveEngine=best.engine --fp16 --memPoolSize=workspace:512M

trtexec --onnx=best.onnx --saveEngine=best.engine --int8 --memPoolSize=workspace:512M

In [8]:
# ============================================================
# 셀8 - .pt 모델 추론 테스트
# ============================================================
from ultralytics import YOLO
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

TEST_DIR   = Path(r"C:\Users\SSAFY\Desktop\03\models\test")
IMG_EXTS   = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SAVE_DIR   = Path(project_dir) / run_name / "pt_test"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
CONF_THRES = 0.25
IOU_THRES  = 0.45

best_model_path = Path(project_dir) / run_name / "weights" / "best.pt"
model = YOLO(str(best_model_path))

img_files = [f for f in TEST_DIR.iterdir() if f.suffix.lower() in IMG_EXTS]
print(f"테스트 이미지 수: {len(img_files)}장")

for img_path in img_files:
    orig = cv2.imread(str(img_path))
    orig_h, orig_w = orig.shape[:2]

    results = model.predict(str(img_path), imgsz=480, conf=CONF_THRES, iou=IOU_THRES, device=0, verbose=False)

    boxes = results[0].boxes
    det_count = len(boxes)

    # 추가 NMS
    if det_count > 0:
        xyxy  = boxes.xyxy.cpu().numpy().astype(int)
        confs = boxes.conf.cpu().numpy()
        x1, y1, x2, y2 = xyxy[:, 0], xyxy[:, 1], xyxy[:, 2], xyxy[:, 3]
        boxes_xywh = np.stack([x1, y1, x2 - x1, y2 - y1], axis=1)
        indices = cv2.dnn.NMSBoxes(boxes_xywh.tolist(), confs.tolist(), CONF_THRES, IOU_THRES)
        indices = indices.flatten() if len(indices) > 0 else []
    else:
        indices = []
        confs   = np.array([])
        x1 = y1 = x2 = y2 = np.array([], dtype=int)

    det_count = len(indices)
    conf_max  = float(confs[indices].max()) if det_count > 0 else 0.0

    vis = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(vis)
    for i in indices:
        rect = patches.Rectangle((x1[i], y1[i]), x2[i]-x1[i], y2[i]-y1[i],
                                   linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1[i], y1[i]-5, f"{confs[i]:.2f}", color='lime', fontsize=9,
                bbox=dict(facecolor='black', alpha=0.4, pad=1, edgecolor='none'))
    ax.axis('off')
    ax.set_title(f"{img_path.name} | 탐지: {det_count}개 | conf max: {conf_max:.4f}")

    save_path = SAVE_DIR / img_path.name
    plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
    plt.close()
    print(f"  [{det_count}탐지 / conf {conf_max:.4f}] {img_path.name} → 저장")

print(f"\n완료. 저장 경로: {SAVE_DIR}")

테스트 이미지 수: 3장


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1574304746.py:62: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1574304746.py:62: UserWarning: Glyph 51648 (\N{HANGUL SYLLABLE JI}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1574304746.py:62: UserWarning: Glyph 44060 (\N{HANGUL SYLLABLE GAE}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1574304746.py:62: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1574304746.py:62: UserWarning: Glyph 51648 (\N{HANGUL SYLLABLE JI}) missing from font(s) DejaVu San

  [1탐지 / conf 0.9062] test (1).jpg → 저장
  [2탐지 / conf 0.7171] test (2).jpg → 저장
  [1탐지 / conf 0.8388] test (3).jpg → 저장

완료. 저장 경로: C:\Users\SSAFY\Desktop\03\models\yolo11n_480_v2\pt_test


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1574304746.py:62: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1574304746.py:62: UserWarning: Glyph 51648 (\N{HANGUL SYLLABLE JI}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1574304746.py:62: UserWarning: Glyph 44060 (\N{HANGUL SYLLABLE GAE}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1574304746.py:62: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1574304746.py:62: UserWarning: Glyph 51648 (\N{HANGUL SYLLABLE JI}) missing from font(s) DejaVu San

In [4]:
# ============================================================
# 셀9 - .onnx 모델 추론 테스트
# ============================================================
import onnxruntime as ort
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

TEST_DIR   = Path(r"C:\Users\SSAFY\Desktop\03\models\test")
IMG_EXTS   = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
ONNX_PATH  = str(Path(project_dir) / run_name / "weights" / "onnx" / "best.onnx")
SAVE_DIR   = Path(project_dir) / run_name / "onnx_test"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
CONF_THRES = 0.25
IOU_THRES  = 0.45
IMG_SIZE   = 480

sess = ort.InferenceSession(ONNX_PATH, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
img_files = [f for f in TEST_DIR.iterdir() if f.suffix.lower() in IMG_EXTS]
print(f"테스트 이미지 수: {len(img_files)}장")

for img_path in img_files:
    orig = cv2.imread(str(img_path))
    orig_h, orig_w = orig.shape[:2]
    img = cv2.resize(orig, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))[np.newaxis]

    output = sess.run(None, {"images": img})[0]
    pred = output[0].T  # (4725, 5)

    mask = pred[:, 4] > CONF_THRES

    vis = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(vis)

    det_count = 0
    conf_max  = 0.0

    if mask.sum() > 0:
        filtered = pred[mask]
        cx, cy, w, h, conf = filtered[:, 0], filtered[:, 1], filtered[:, 2], filtered[:, 3], filtered[:, 4]
        scale_x = orig_w / IMG_SIZE
        scale_y = orig_h / IMG_SIZE
        x1 = ((cx - w / 2) * scale_x).astype(int)
        y1 = ((cy - h / 2) * scale_y).astype(int)
        x2 = ((cx + w / 2) * scale_x).astype(int)
        y2 = ((cy + h / 2) * scale_y).astype(int)

        # NMS
        boxes_xywh = np.stack([x1, y1, x2 - x1, y2 - y1], axis=1)
        indices = cv2.dnn.NMSBoxes(boxes_xywh.tolist(), conf.tolist(), CONF_THRES, IOU_THRES)
        indices = indices.flatten() if len(indices) > 0 else []

        det_count = len(indices)
        conf_max  = float(conf[indices].max()) if det_count > 0 else 0.0

        for i in indices:
            rect = patches.Rectangle((x1[i], y1[i]), x2[i]-x1[i], y2[i]-y1[i],
                                       linewidth=2, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1[i], y1[i]-5, f"{conf[i]:.2f}", color='lime', fontsize=9,
                    bbox=dict(facecolor='black', alpha=0.4, pad=1, edgecolor='none'))

    ax.axis('off')
    ax.set_title(f"{img_path.name} | 탐지: {det_count}개 | conf max: {conf_max:.4f}")

    save_path = SAVE_DIR / img_path.name
    plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
    plt.close()
    print(f"  [{det_count}탐지 / conf {conf_max:.4f}] {img_path.name} → 저장")

print(f"\n완료. 저장 경로: {SAVE_DIR}")


c:\Users\SSAFY\miniforge3\envs\yolo11\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1099374494.py:72: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1099374494.py:72: UserWarning: Glyph 51648 (\N{HANGUL SYLLABLE JI}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1099374494.py:72: UserWarning: Glyph 44060 (\N{HANGUL SYLLABLE GAE}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1099374494.py:72: User

테스트 이미지 수: 3장
  [1탐지 / conf 0.8848] test (1).jpg → 저장


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1099374494.py:72: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1099374494.py:72: UserWarning: Glyph 51648 (\N{HANGUL SYLLABLE JI}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1099374494.py:72: UserWarning: Glyph 44060 (\N{HANGUL SYLLABLE GAE}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1099374494.py:72: UserWarning: Glyph 53456 (\N{HANGUL SYLLABLE TAM}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_17504\1099374494.py:72: UserWarning: Glyph 51648 (\N{HANGUL SYLLABLE JI}) missing from font(s) DejaVu San

  [2탐지 / conf 0.8388] test (2).jpg → 저장
  [1탐지 / conf 0.7612] test (3).jpg → 저장

완료. 저장 경로: C:\Users\SSAFY\Desktop\03\models\yolo11n_480_v2\onnx_test
